In [1]:
from pathlib import Path
import re
import pandas as pd

from ase.io import read, write
from ase.io.gaussian import read_gaussian_out

from export_mlopt_xyz_to_total import mlopt_to_total
from parse_dftsp_grepscf_to_df import parse_grepscf

hartree_to_eV = 27.2114
hartree_to_kJpmol = 2625.5096


### initialization

In [ ]:
# # cp ml opt .xyz to opt_structure_total/
# temp1 = mlopt_to_total(8)

In [ ]:
# # extract dft opt from .log to .xyz
# dft_opt_folder = Path("./tio2_ChenDixon_gjf/8_original")
# save_folder = Path("./opt_structures_total")

# for file_i in list(dft_opt_folder.glob("8*.log")):
#     # read coordination
#     print(file_i)
#     fcontent_i = open(file_i, "r")
#     atoms_i = read_gaussian_out(fcontent_i, index=-1)

#     # xyz file name
#     fname_i = file_i.stem.split("_")
#     fstructure_i = fname_i[0]
#     if fname_i[1] == "pbe":
#         fmethod_i = "pbe_def2svp"
#     else:
#         fmethod_i = "pbe0_def2svp"
#     save_name = fstructure_i + "_" + fmethod_i + ".xyz"
#     print(save_name)

#     # write xyz
#     save_path = save_folder / save_name
#     atoms_i.write(save_path, format="xyz")



In [2]:
# read all dft/tzvp sp energies to df
sp_E_path = Path("./tio2_ChenDixon_gjf/all8_sp/all8_sp_energies.txt")

df_sp_E = parse_grepscf(sp_E_path)
df_sp_E

,structure,opt method,sp method,energy_au
0,8a,matpes_pbe,pbe0_def2tzvp,-7998.675239
1,8a,matpes_r2scan,pbe0_def2tzvp,-7998.684010
2,8a,mpa_mpa,pbe0_def2tzvp,-7998.675920
3,8a,omat24_omat24,pbe0_def2tzvp,-7998.675645
4,8a,omol25_low,pbe0_def2tzvp,-7998.689734
5,8a,original_original,pbe0_def2tzvp,-7998.680535
6,8a,pbe0_def2svp,pbe0_def2tzvp,-7998.689023
7,8a,pbe_def2svp,pbe0_def2tzvp,-7998.688561
8,8a,matpes_pbe,pbe_def2tzvp,-7998.786960
9,8a,matpes_r2scan,pbe_def2tzvp,-7998.788403


# sp energies

In [3]:
df_sp_E

,structure,opt method,sp method,energy_au
0,8a,matpes_pbe,pbe0_def2tzvp,-7998.675239
1,8a,matpes_r2scan,pbe0_def2tzvp,-7998.684010
2,8a,mpa_mpa,pbe0_def2tzvp,-7998.675920
3,8a,omat24_omat24,pbe0_def2tzvp,-7998.675645
4,8a,omol25_low,pbe0_def2tzvp,-7998.689734
5,8a,original_original,pbe0_def2tzvp,-7998.680535
6,8a,pbe0_def2svp,pbe0_def2tzvp,-7998.689023
7,8a,pbe_def2svp,pbe0_def2tzvp,-7998.688561
8,8a,matpes_pbe,pbe_def2tzvp,-7998.786960
9,8a,matpes_r2scan,pbe_def2tzvp,-7998.788403


In [4]:
df_pbe0sp_E = df_sp_E[df_sp_E["sp method"] == "pbe0_def2tzvp"]
df_pbesp_E = df_sp_E[df_sp_E["sp method"] == "pbe_def2tzvp"]
df_pbe0sp_E

,structure,opt method,sp method,energy_au
0,8a,matpes_pbe,pbe0_def2tzvp,-7998.675239
1,8a,matpes_r2scan,pbe0_def2tzvp,-7998.684010
2,8a,mpa_mpa,pbe0_def2tzvp,-7998.675920
3,8a,omat24_omat24,pbe0_def2tzvp,-7998.675645
4,8a,omol25_low,pbe0_def2tzvp,-7998.689734
5,8a,original_original,pbe0_def2tzvp,-7998.680535
6,8a,pbe0_def2svp,pbe0_def2tzvp,-7998.689023
7,8a,pbe_def2svp,pbe0_def2tzvp,-7998.688561
16,8b,matpes_pbe,pbe0_def2tzvp,-7998.674781
17,8b,matpes_r2scan,pbe0_def2tzvp,-7998.683693


In [5]:
def reformat_df(df: pd.DataFrame, ref: str):
    df = df.reset_index(drop=True)
    df = df.drop("sp method", axis=1)
    df_a = df.loc[df['structure'] == '8a', :]
    df_b = df.loc[df['structure'] == '8b', :]
    df_b = df_b.drop(["structure", "opt method"], axis=1)
    df_b = df_b.reset_index(drop=True)
    df_out = pd.concat([df_a, df_b], axis=1)
    df_out.columns.values[2] = "8a au"
    df_out.columns.values[3] = "8b au"
    df_out = df_out.drop("structure", axis=1)

    df_out.loc[:, "dE eV"] = (df_out.loc[:, "8b au"] - df_out.loc[:, "8a au"]) * hartree_to_eV

    ref_8a = df_out.loc[df_out['opt method'] == ref, "8a au"].iloc[0]
    ref_8b = df_out.loc[df_out['opt method'] == ref, "8b au"].iloc[0]
    print(ref_8a)
    print(ref_8b)
    df_out.loc[:, "8a ref eV"] = (df_out.loc[:, "8a au"] - ref_8a) * hartree_to_eV
    df_out.loc[:, "8b ref eV"] = (df_out.loc[:, "8b au"] - ref_8b) * hartree_to_eV
    df_out.loc[:, "dE ref eV"] = df_out.loc[:, "8b ref eV"] - df_out.loc[:, "8a ref eV"]

    return df_out

reformat_df(df_pbe0sp_E, "pbe0_def2svp")

-7998.68902269
-7998.6891355


,opt method,8a au,8b au,dE eV,8a ref eV,8b ref eV,dE ref eV
0,matpes_pbe,-7998.675239,-7998.674781,0.012465,0.375080,0.390615,0.015535
1,matpes_r2scan,-7998.684010,-7998.683693,0.008611,0.136411,0.148092,0.011681
2,mpa_mpa,-7998.675920,-7998.675791,0.003506,0.356541,0.363117,0.006576
3,omat24_omat24,-7998.675645,-7998.675534,0.003022,0.364028,0.370119,0.006091
4,omol25_low,-7998.689734,-7998.689868,-0.003636,-0.019362,-0.019928,-0.000566
5,original_original,-7998.680535,-7998.680393,0.003873,0.230953,0.237896,0.006943
6,pbe0_def2svp,-7998.689023,-7998.689135,-0.003070,0.000000,0.000000,0.000000
7,pbe_def2svp,-7998.688561,-7998.688511,0.001359,0.012572,0.017001,0.004429


In [6]:
reformat_df(df_pbesp_E, "pbe_def2svp")

-7998.78711467
-7998.78696145


,opt method,8a au,8b au,dE eV,8a ref eV,8b ref eV,dE ref eV
0,matpes_pbe,-7998.786960,-7998.786760,0.005431,0.004219,0.005481,0.001261
1,matpes_r2scan,-7998.788403,-7998.788164,0.006515,-0.035070,-0.032724,0.002346
2,mpa_mpa,-7998.787075,-7998.787039,0.000981,0.001076,-0.002112,-0.003189
3,omat24_omat24,-7998.787062,-7998.786959,0.002797,0.001428,0.000055,-0.001373
4,omol25_low,-7998.784245,-7998.784818,-0.015580,0.078085,0.058335,-0.019750
5,original_original,-7998.787842,-7998.787465,0.010268,-0.019787,-0.013689,0.006098
6,pbe0_def2svp,-7998.774426,-7998.773949,0.012980,0.345268,0.354078,0.008810
7,pbe_def2svp,-7998.787115,-7998.786961,0.004169,0.000000,0.000000,0.000000
